In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Display more columns when inspecting the data
pd.set_option("display.max_columns", None)

# Load the dataset
df = pd.read_csv("crime-housing-austin-2015.csv")

# Inspect the data
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 38573
Columns: 43


,Key,Council_District,Highest_Offense_Desc,Highest_NIBRS_UCR_Offense_Description,Report_Date,Location,Clearance_Status,Clearance_Date,District,Zip_Code_Crime,Census_Tract,X_Coordinate,Y_Coordinate,Zip_Code_Housing,Populationbelowpovertylevel,Medianhouseholdincome,Non-WhiteNon-HispanicorLatino,HispanicorLatinoofanyrace,Populationwithdisability,Unemployment,Largehouseholds(5+members),Homesaffordabletopeopleearninglessthan$50000,Rentalsaffordabletopeopleearninglessthan$25000,Rent-restrictedunits,HousingChoiceVoucherholders,Medianrent,Medianhomevalue,Percentageofrentalunitsinpoorcondition,Percentchangeinnumberofhousingunits2000-2012,Ownerunitsaffordabletoaverageretail/serviceworker,Rentalunitsaffordabletoaverageretail/serviceworker,Rentalunitsaffordabletoaverageartist,Ownerunitsaffordabletoaverageartist,Rentalunitsaffordabletoaverageteacher,Ownerunitsaffordabletoaverageteacher,Rentalunitsaffordabletoaveragetechworker,Ownerunitsaffordabletoaveragetechworker,Changeinpercentageofpopulationbelowpoverty2000-2012,Changeinmedianrent2000-2012,Changeinmedianhomevalue2000-2012,Percentageofhomeswithin1/4-mioftransitstop,Averagemonthlytransportationcost,Percentageofhousingandtransportationcoststhatistransportation-related
0,201510782,4.0,AGG ROBBERY/DEADLY WEAPON,Robbery,1-Jan-15,9001 N IH 35 SVRD NB,N,28-Jan-15,E,78753.0,18.13,3130483.0,10102366.0,78753.0,26%,$39593,20%,60%,10%,9%,16%,78%,14%,17%,4%,$826,$134900,0.7%,19%,8%,11%,40%,24%,89%,75%,100%,98%,128%,26%,40%,59%,$708,44%
1,201511231,4.0,ROBBERY BY ASSAULT,Robbery,1-Jan-15,919 E KOENIG LN SVRD EB,N,13-Jan-15,I,78751.0,21.05,3124730.0,10090296.0,78751.0,26%,$38624,11%,14%,6%,9%,2%,11%,13%,1%,0%,$865,$292200,0.4%,7%,0%,9%,38%,2%,68%,10%,97%,42%,23%,38%,97%,98%,$550,40%
2,201511736,1.0,BURGLARY OF RESIDENCE,Burglary,1-Jan-15,12151 N IH 35 SVRD NB,N,13-Jan-15,E,78753.0,18.35,3135985.0,10117220.0,78753.0,26%,$39593,20%,60%,10%,9%,16%,78%,14%,17%,4%,$826,$134900,0.7%,19%,8%,11%,40%,24%,89%,75%,100%,98%,128%,26%,40%,59%,$708,44%
3,201511433,4.0,BURGLARY OF RESIDENCE,Burglary,1-Jan-15,1044 NORWOOD PARK BLVD,N,5-Jan-15,I,78753.0,18.13,3129896.0,10096032.0,78753.0,26%,$39593,20%,60%,10%,9%,16%,78%,14%,17%,4%,$826,$134900,0.7%,19%,8%,11%,40%,24%,89%,75%,100%,98%,128%,26%,40%,59%,$708,44%
4,201511936,2.0,BURGLARY OF RESIDENCE,Burglary,1-Jan-15,2413 BITTER CREEK DR,N,7-Jan-15,F,78744.0,24.27,3110455.0,10039340.0,78744.0,26%,$41056,9%,77%,8%,9%,23%,93%,7%,24%,8%,$946,$108100,0.7%,35%,13%,6%,22%,33%,81%,87%,100%,100%,89%,26%,44%,63%,$708,40%


In [2]:
# Make a copy so the original dataframe remains unchanged
data = df.copy()

# Convert ZIP Code to numeric
data["Zip_Code_Crime"] = pd.to_numeric(
    data["Zip_Code_Crime"], errors="coerce"
)

data["Zip_Code_Housing"] = pd.to_numeric(
    data["Zip_Code_Housing"], errors="coerce"
)

# Convert socioeconomic/housing variables to numeric
numeric_columns = [
    "Populationbelowpovertylevel",
    "Medianhouseholdincome",
    "Unemployment",
    "Medianrent",
    "Medianhomevalue",
    "Percentageofrentalunitsinpoorcondition",
    "Homesaffordabletopeopleearninglessthan$50000",
    "Averagemonthlytransportationcost",
    "Percentageofhousingandtransportationcoststhatistransportation-related"
]

for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Check missing values
data[numeric_columns + ["Zip_Code_Crime", "Zip_Code_Housing"]].isna().sum()


Populationbelowpovertylevel                                              38573
Medianhouseholdincome                                                    38573
Unemployment                                                             38573
Medianrent                                                               38573
Medianhomevalue                                                          38573
Percentageofrentalunitsinpoorcondition                                   38573
Homesaffordabletopeopleearninglessthan$50000                             38573
Averagemonthlytransportationcost                                         38573
Percentageofhousingandtransportationcoststhatistransportation-related    38573
Zip_Code_Crime                                                             159
Zip_Code_Housing                                                          1288
dtype: int64

In [3]:
print("Unique crime ZIP Codes:", data["Zip_Code_Crime"].nunique())
print("Unique housing ZIP Codes:", data["Zip_Code_Housing"].nunique())

print("\nExample crime ZIP Codes:")
print(sorted(data["Zip_Code_Crime"].dropna().unique())[:20])

print("\nExample housing ZIP Codes:")
print(sorted(data["Zip_Code_Housing"].dropna().unique())[:20])

Unique crime ZIP Codes: 47
Unique housing ZIP Codes: 36

Example crime ZIP Codes:
[np.float64(78613.0), np.float64(78617.0), np.float64(78652.0), np.float64(78653.0), np.float64(78660.0), np.float64(78701.0), np.float64(78702.0), np.float64(78703.0), np.float64(78704.0), np.float64(78705.0), np.float64(78712.0), np.float64(78717.0), np.float64(78719.0), np.float64(78721.0), np.float64(78722.0), np.float64(78723.0), np.float64(78724.0), np.float64(78725.0), np.float64(78726.0), np.float64(78727.0)]

Example housing ZIP Codes:
[np.float64(78617.0), np.float64(78701.0), np.float64(78702.0), np.float64(78703.0), np.float64(78704.0), np.float64(78705.0), np.float64(78717.0), np.float64(78721.0), np.float64(78722.0), np.float64(78723.0), np.float64(78724.0), np.float64(78726.0), np.float64(78727.0), np.float64(78728.0), np.float64(78729.0), np.float64(78730.0), np.float64(78731.0), np.float64(78732.0), np.float64(78735.0), np.float64(78739.0)]


In [4]:
# Count reported crime records in each ZIP Code
crime_counts = (
    data.dropna(subset=["Zip_Code_Crime"])
        .groupby("Zip_Code_Crime")
        .size()
        .reset_index(name="Crime_Count")
)

crime_counts.head()
crime_counts["Crime_Count"].describe()
crime_counts.sort_values(
    "Crime_Count", ascending=False
).head(10)

,Zip_Code_Crime,Crime_Count
41,78753.0,3472
30,78741.0,2973
8,78704.0,2571
45,78758.0,2563
33,78745.0,2422
15,78723.0,2124
5,78701.0,2103
32,78744.0,1921
6,78702.0,1668
36,78748.0,1536


In [5]:
data.groupby("Zip_Code_Housing")["Medianhouseholdincome"].nunique().describe()
housing_variables = [
    "Populationbelowpovertylevel",
    "Medianhouseholdincome",
    "Unemployment",
    "Medianrent",
    "Medianhomevalue",
    "Percentageofrentalunitsinpoorcondition",
    "Homesaffordabletopeopleearninglessthan$50000",
    "Averagemonthlytransportationcost",
    "Percentageofhousingandtransportationcoststhatistransportation-related"
]

zip_housing = (
    data.dropna(subset=["Zip_Code_Housing"])
        .groupby("Zip_Code_Housing")[housing_variables]
        .median()
        .reset_index()
)

zip_housing.head()

,Zip_Code_Housing,Populationbelowpovertylevel,Medianhouseholdincome,Unemployment,Medianrent,Medianhomevalue,Percentageofrentalunitsinpoorcondition,Homesaffordabletopeopleearninglessthan$50000,Averagemonthlytransportationcost,Percentageofhousingandtransportationcoststhatistransportation-related
0,78617.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,78701.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,78702.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,78703.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,78704.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
